# Proyecto práctico final: Prototipo inteligente para AI Elite, S.R.L.

Este notebook presenta el desarrollo de un prototipo inteligente básico para la empresa AI Elite, S.R.L., una organización del sector educativo que recibe solicitudes de soporte técnico relacionadas con acceso al sistema, correo electrónico, internet, impresoras, equipos lentos, bases de datos y plataformas digitales.

El objetivo del prototipo es apoyar el análisis de solicitudes de soporte mediante Python, estructuras de datos, reglas de decisión, reconocimiento de patrones, un agente inteligente, un chatbot básico y un modelo de Machine Learning para predecir la prioridad de los casos.

In [ ]:
# ============================================================
# Proyecto Final - ISW-309
# AI Elite, S.R.L.
# Sistema inteligente para gestión de solicitudes de soporte
# Autor: Franklin Beltré
# Matricula: 100066613
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

print("Librerías cargadas correctamente.")

## Carga del dataset

En esta sección se carga el archivo solicitudes_soporte_ia.csv, el cual contiene solicitudes simuladas de soporte técnico recibidas por AI Elite, S.R.L.

Cada registro representa un ticket de soporte y contiene información como categoría del problema, área afectada, urgencia, impacto, canal de recepción, tipo de usuario, dispositivo, horario, prioridad asignada, necesidad de escalamiento y acción recomendada.

In [ ]:
# Cargar el dataset desde GitHub

url_raw =("https://raw.githubusercontent.com/100066613/Proyecto-Final-IA/main/solicitudes_soporte_ia.csv")

df = pd.read_csv(url_raw)

# Mostrar las primeras filas
df.head()

## Exploración inicial de los datos

Antes de construir cualquier componente inteligente, es necesario conocer la estructura del dataset. En esta parte se revisa la cantidad de registros, las columnas disponibles, el tipo de dato de cada columna y la existencia de valores faltantes.

In [ ]:
# Columnas disponibles
df.shape

In [ ]:
# Columnas disponibles
df.columns

In [ ]:
# Información general del dataset
df.info()

In [ ]:
# Resumen descriptivo del conjunto de datos
df.describe()

In [ ]:
# Revisión de valores faltantes
df.isnull().sum()

### Interpretación de la exploración inicial

El dataset contiene 90 registros y 14 columnas. No se encontraron valores faltantes, lo que permite trabajar con los datos sin aplicar una limpieza profunda. La variable principal para el modelo de Machine Learning será prioridad, ya que representa la clasificación que se desea predecir: Baja, Media o Alta.

In [ ]:
# Distribución de la variable objetivo
df["prioridad"].value_counts()

# Representación computacional del problema

Antes de desarrollar cualquier componente inteligente, es importante representar el problema desde el punto de vista computacional.

En este proyecto, cada fila del dataset representa una solicitud de soporte técnico (ticket), mientras que cada columna almacena información relevante para describir ese caso, como la categoría del problema, el área afectada, la urgencia, el impacto, el tipo de usuario, el dispositivo involucrado y la prioridad asignada.

Esta representación permite que el computador procese la información mediante estructuras de datos y algoritmos para realizar búsquedas, aplicar reglas de decisión, identificar patrones y entrenar modelos de Machine Learning capaces de apoyar la toma de decisiones.

## Variables del problema

Las variables disponibles en el dataset son:

- Ticket
- Fecha del reporte
- Categoría
- Área
- Urgencia
- Impacto
- Tiempo estimado
- Canal
- Tipo de usuario
- Dispositivo
- Horario
- Prioridad
- Escalar
- Acción recomendada

La variable objetivo del proyecto será **prioridad**, ya que el modelo de Machine Learning intentará predecir si una nueva solicitud debe clasificarse como Baja, Media o Alta.

In [ ]:
# Mostrar las columnas del dataset de forma organizada

for columna in df.columns:
    print("-", columna)

### Interpretación

La representación del problema permite identificar claramente cuáles variables describen cada solicitud y cuál será la variable que el modelo deberá predecir. Esta organización será utilizada posteriormente en el sistema de búsqueda, el agente inteligente, el chatbot y el modelo de Machine Learning.

# Búsqueda inteligente por número de ticket

Una de las necesidades principales de AI Elite, S.R.L. es poder consultar rápidamente una solicitud de soporte técnico mediante su número de ticket.

En esta sección se implementa una función de búsqueda que permite localizar un caso específico dentro del dataset. Si el ticket existe, el sistema muestra la información organizada como un reporte de soporte. Si el ticket no existe, se presenta un mensaje claro indicando que no fue encontrado.

In [ ]:
def buscar_ticket(numero_ticket):
    try:
        numero_ticket = int(numero_ticket)
    except ValueError:
        print("=" * 70)
        print("El número de ticket debe ser un valor numérico.")
        print("=" * 70)
        return

    resultado = df[df["ticket"] == numero_ticket]

    if resultado.empty:
        print("=" * 70)
        print("No se encontró ningún ticket con ese número.")
        print("=" * 70)
    else:
        ticket = resultado.iloc[0]

        print("=" * 70)
        print("        AI Elite, S.R.L.")
        print("        Sistema Inteligente de Soporte Técnico")
        print("=" * 70)
        print("Consulta realizada correctamente\n")

        print(f"Ticket................... {ticket['ticket']}")
        print(f"Fecha.................... {ticket['fecha_reporte']}")
        print(f"Categoría................ {ticket['categoria']}")
        print(f"Área..................... {ticket['area']}")
        print(f"Urgencia................. {ticket['urgencia']}")
        print(f"Impacto.................. {ticket['impacto']}")
        print(f"Tiempo estimado.......... {ticket['tiempo_estimado_min']} minutos")
        print(f"Canal.................... {ticket['canal']}")
        print(f"Tipo de usuario.......... {ticket['tipo_usuario']}")
        print(f"Dispositivo.............. {ticket['dispositivo']}")
        print(f"Horario.................. {ticket['horario']}")
        print(f"Prioridad................ {ticket['prioridad']}")
        print(f"Escalar.................. {ticket['escalar']}")
        print(f"Acción recomendada....... {ticket['accion_recomendada']}")
        print("=" * 70)

In [ ]:
# Ejemplo de búsqueda existente

buscar_ticket(1008)


In [ ]:
# Ejemplo de búsqueda inexistente

buscar_ticket(9999)


### Interpretación de la búsqueda

La función de búsqueda permite consultar una solicitud de soporte mediante el número de ticket. Cuando el ticket existe, el sistema muestra los datos principales del caso en un formato organizado, lo cual facilita la revisión rápida de la incidencia.

Cuando el ticket no existe, el sistema informa que no se encontró ningún registro asociado. Esta validación es importante porque evita mostrar resultados vacíos o confusos al usuario.

# Sistema básico de reglas y decisiones

AI Elite, S.R.L. necesita priorizar las solicitudes de soporte técnico para evitar retrasos en los casos más críticos. En esta sección se implementa un sistema básico de reglas que analiza variables como urgencia, impacto y necesidad de escalamiento para recomendar una acción inicial.

Este sistema no reemplaza al personal técnico, pero puede servir como apoyo para orientar la primera respuesta ante una incidencia.

In [ ]:
def recomendar_accion(urgencia, impacto, escalar):
    urgencia = urgencia.lower()
    impacto = impacto.lower()
    escalar = escalar.lower()

    if urgencia == "alta" and impacto == "alto":
        return "Atención inmediata y escalamiento al equipo técnico especializado."

    elif urgencia == "alta" and escalar == "sí":
        return "Revisar el caso con prioridad alta y escalar si afecta servicios críticos."

    elif urgencia == "media":
        return "Atender durante el día y dar seguimiento al usuario afectado."

    elif urgencia == "baja":
        return "Registrar el caso para seguimiento posterior según disponibilidad del equipo."

    else:
        return "Revisar manualmente la solicitud para determinar la acción adecuada."

In [ ]:
# Prueba del sistema básico de reglas

caso_prueba = df.iloc[6]

print("=" * 70)
print("EVALUACIÓN DEL SISTEMA DE REGLAS")
print("=" * 70)

print(f"Ticket:              {caso_prueba['ticket']}")
print(f"Categoría:           {caso_prueba['categoria']}")
print(f"Urgencia:            {caso_prueba['urgencia']}")
print(f"Impacto:             {caso_prueba['impacto']}")
print(f"Escalar:             {caso_prueba['escalar']}")

accion = recomendar_accion(
    caso_prueba["urgencia"],
    caso_prueba["impacto"],
    caso_prueba["escalar"]
)

print("\nAcción recomendada por el sistema:")
print(accion)

### Interpretación del sistema de reglas

El sistema de reglas permite generar una recomendación inicial a partir de condiciones simples. Si una solicitud presenta urgencia alta e impacto alto, el sistema recomienda atención inmediata y escalamiento. Si la urgencia es media o baja, la recomendación se ajusta al nivel de prioridad del caso.

Este componente representa una primera forma de razonamiento automatizado dentro del prototipo, ya que transforma datos del ticket en una acción sugerida.

# Reconocimiento de patrones

El reconocimiento de patrones consiste en identificar comportamientos, tendencias o relaciones presentes en un conjunto de datos.

En AI Elite, S.R.L. este análisis permite conocer cuáles categorías generan más incidencias, qué niveles de prioridad aparecen con mayor frecuencia, qué áreas reciben más solicitudes de soporte y cuáles son los dispositivos que presentan más problemas.

Esta información resulta útil para apoyar la toma de decisiones y planificar acciones de mejora dentro de la organización.

In [ ]:
print("="*70)
print("        AI Elite, S.R.L.")
print("   Resumen general del dataset")
print("="*70)

print(f"Total de tickets............... {len(df)}")
print(f"Categorías..................... {df['categoria'].nunique()}")
print(f"Áreas.......................... {df['area'].nunique()}")
print(f"Tipos de usuario............... {df['tipo_usuario'].nunique()}")
print(f"Dispositivos................... {df['dispositivo'].nunique()}")

print("="*70)

### Interpretación

El conjunto de datos contiene 90 solicitudes de soporte técnico distribuidas entre diferentes categorías, áreas, tipos de usuario y dispositivos. Esta diversidad permitirá identificar patrones que posteriormente podrán ser utilizados por el sistema de reglas, el agente inteligente y el modelo de Machine Learning.

## Distribución de prioridades

Uno de los primeros análisis consiste en conocer cómo se distribuyen las prioridades de las solicitudes registradas.

In [ ]:
prioridades = df["prioridad"].value_counts()

print(prioridades)

In [ ]:
prioridades.plot(kind="bar")

plt.title("Distribución de prioridades")

plt.xlabel("Prioridad")

plt.ylabel("Cantidad")

plt.show()

### Interpretación

El análisis permite identificar las categorías que generan un mayor número de solicitudes de soporte. Esta información puede ayudar a la empresa a enfocar esfuerzos de capacitación, mantenimiento preventivo o mejoras tecnológicas en las áreas donde se presentan más incidencias.

## Categorías con mayor cantidad de incidencias

En esta sección se analiza cuáles tipos de problemas aparecen con mayor frecuencia dentro del conjunto de datos.

In [ ]:
categorias = df["categoria"].value_counts()

print(categorias)

In [ ]:
categorias.plot(kind="bar")

plt.title("Solicitudes por categoría")

plt.xlabel("Categoría")

plt.ylabel("Cantidad")

plt.xticks(rotation=45)

plt.show()

### Interpretación

El análisis permite identificar las categorías que generan un mayor número de solicitudes de soporte. Esta información puede ayudar a la empresa a enfocar esfuerzos de capacitación, mantenimiento preventivo o mejoras tecnológicas en las áreas donde se presentan más incidencias.

## Áreas que generan más solicitudes

También resulta importante conocer cuáles áreas de la organización reportan la mayor cantidad de incidencias.

In [ ]:
areas = df["area"].value_counts()

print(areas)

In [ ]:
areas.plot(kind="bar")

plt.title("Solicitudes por área")

plt.xlabel("Área")

plt.ylabel("Cantidad")

plt.xticks(rotation=45)

plt.show()

### Interpretación

La distribución por áreas permite identificar los departamentos que requieren mayor atención por parte del equipo de soporte técnico. Esta información puede utilizarse para planificar recursos, fortalecer la infraestructura tecnológica y reducir la aparición de incidencias recurrentes.

## Dispositivos con mayor número de incidencias

Finalmente, se analiza qué tipo de dispositivo aparece con mayor frecuencia dentro de las solicitudes registradas.

In [ ]:
dispositivos = df["dispositivo"].value_counts()

print(dispositivos)

In [ ]:
dispositivos.plot(kind="bar")

plt.title("Incidencias por dispositivo")

plt.xlabel("Dispositivo")

plt.ylabel("Cantidad")

plt.xticks(rotation=45)

plt.show()

### Interpretación

Este análisis permite identificar cuáles dispositivos presentan una mayor cantidad de incidencias dentro de la organización. La información obtenida puede servir como apoyo para definir estrategias de mantenimiento preventivo y renovación de equipos.

# Agente inteligente SupportSmart

El siguiente componente representa un agente inteligente desarrollado para AI Elite, S.R.L.

Su función consiste en analizar automáticamente la información de una solicitud de soporte técnico, interpretar las características del problema y recomendar una acción inicial que apoye al personal del departamento de tecnología.

El agente sigue la arquitectura estudiada durante la asignatura:

- Percibe información del entorno.
- Analiza esa información.
- Aplica reglas de decisión.
- Ejecuta una acción.
- Justifica la decisión tomada.

Aunque este agente utiliza reglas previamente definidas y no aprende automáticamente, representa el funcionamiento básico de un sistema inteligente orientado a la toma de decisiones.

In [ ]:
class SupportSmartIA:

    def analizar_ticket(self, ticket):

        urgencia = ticket["urgencia"]
        impacto = ticket["impacto"]
        escalar = ticket["escalar"]
        horario = ticket["horario"]

        print("="*70)
        print("              AI Elite, S.R.L.")
        print("      Agente inteligente SupportSmart")
        print("="*70)

        print("\n1. PERCEPCIÓN DEL ENTORNO")
        print("-"*70)

        print(f"Ticket................ {ticket['ticket']}")
        print(f"Categoría............. {ticket['categoria']}")
        print(f"Área.................. {ticket['area']}")
        print(f"Urgencia.............. {urgencia}")
        print(f"Impacto............... {impacto}")
        print(f"Horario............... {horario}")
        print(f"Escalar............... {escalar}")

        print("\n2. ANÁLISIS")

        reglas = []

        if urgencia == "Alta":
            reglas.append("Urgencia alta detectada")

        if impacto == "Alto":
            reglas.append("Impacto alto detectado")

        if escalar == "Sí":
            reglas.append("El caso requiere escalamiento")

        if horario == "Fuera de horario":
            reglas.append("Incidencia fuera del horario laboral")

        print("-"*70)

        for regla in reglas:
            print("✓", regla)

        print("\n3. TOMA DE DECISIÓN")
        print("-"*70)

        if urgencia == "Alta" and impacto == "Alto":

            decision = "ATENCIÓN INMEDIATA"

            accion = "Asignar inmediatamente al equipo especializado."

        elif urgencia == "Alta":

            decision = "ATENCIÓN PRIORITARIA"

            accion = "Asignar al siguiente técnico disponible."

        elif urgencia == "Media":

            decision = "ATENCIÓN PROGRAMADA"

            accion = "Atender durante la jornada laboral."

        else:

            decision = "SEGUIMIENTO"

            accion = "Programar revisión posterior."

        if escalar == "Sí":

            decision += " + ESCALAR"

        print(f"Decisión.............. {decision}")

        print("\n4. ACCIÓN RECOMENDADA")
        print("-"*70)

        print(accion)

        print("\n5. JUSTIFICACIÓN")

        print("-"*70)

        print(
            "La decisión fue tomada considerando la combinación "
            "de urgencia, impacto y necesidad de escalamiento "
            "registradas en la solicitud."
        )

        print("="*70)

In [ ]:
agente = SupportSmartIA()

ticket = df.iloc[12]

agente.analizar_ticket(ticket)

### Interpretación

El agente inteligente recibe la información contenida en un ticket de soporte y la procesa siguiendo una secuencia organizada de pasos. Primero identifica las características principales de la solicitud, posteriormente aplica un conjunto de reglas de decisión y finalmente genera una recomendación junto con una explicación del motivo por el cual fue seleccionada.

Este comportamiento permite representar el funcionamiento básico de un agente inteligente, demostrando cómo un sistema puede percibir información del entorno, analizarla y actuar para alcanzar un objetivo específico.

# Chatbot SupportBot con consulta inteligente de tickets

En esta sección se implementa el chatbot **SupportBot**, diseñado como un canal de atención inicial para los usuarios que registran solicitudes de soporte técnico en **AI Elite, S.R.L.** Su objetivo es responder consultas frecuentes relacionadas con el servicio de soporte y permitir que los usuarios consulten información sobre sus solicitudes sin necesidad de comunicarse directamente con el personal técnico.

El chatbot combina una base de conocimiento con la consulta dinámica del dataset utilizado durante el proyecto. Gracias a esta integración, puede responder preguntas generales sobre el proceso de soporte y recuperar información específica de los tickets cuando el usuario proporciona su número de identificación.

Entre las consultas que puede responder se encuentran:

- ¿Cuál es el estado del ticket 1007?
- ¿Qué prioridad tiene el ticket 1007?
- ¿Debe escalarse el ticket 1007?
- ¿Qué acción recomienda el sistema para el ticket 1007?
- ¿Cuál es la categoría del ticket 1007?
- Muéstrame la información del ticket 1007.

Para ofrecer estas respuestas, el chatbot identifica automáticamente el número del ticket dentro de la consulta del usuario, localiza el registro correspondiente en el dataset y genera una respuesta utilizando lenguaje natural.

Con este enfoque, **SupportBot** actúa como un asistente virtual de consulta para los usuarios del sistema de soporte, proporcionando información inmediata sobre sus solicitudes y reduciendo la necesidad de realizar consultas repetitivas al departamento de soporte técnico. Mientras tanto, la asignación y distribución de los tickets continúa realizándose mediante los procedimientos habituales de la organización, de acuerdo con la prioridad asignada y el orden de llegada de las incidencias.

# Diseño del chatbot

**Nombre:** SupportBot

**Objetivo**

Brindar orientación inicial a los usuarios y permitir la consulta de información sobre las solicitudes de soporte registradas en el sistema.

**Base de conocimiento**

El chatbot responde consultas relacionadas con:

- Prioridades
- Categorías
- Tickets
- Escalamiento
- Agente inteligente
- Machine Learning
- Soporte técnico
- AI Elite, S.R.L.

**Tecnologías utilizadas**

- Python
- Pandas
- Expresiones regulares (`re`)
- Búsqueda aproximada (`difflib`)

**Funcionamiento**

1. Recibe la consulta del usuario.
2. Detecta si la consulta contiene un número de ticket.
3. Si encuentra un ticket, consulta el dataset y recupera la información correspondiente.
4. Si la consulta no está relacionada con un ticket, busca la respuesta en la base de conocimiento.
5. Si no existen coincidencias exactas, realiza una búsqueda aproximada utilizando la biblioteca **difflib**.
6. Genera una respuesta utilizando lenguaje natural.
7. Si no encuentra información suficiente para responder, informa al usuario que la consulta no puede ser atendida.

se uso difflib porque es perfecta para manejar los errores de dedo o las distintas formas en que la gente escribe. Si un usuario escribe con una falta de ortografía o cambia una palabra, la librería es capaz de medir el 'parecido' entre frases y entender la intención real. Nos da un chatbot flexible y amigable sin necesidad de meterse en algoritmos complejos.

In [ ]:
# ============================================================
# Chatbot SupportBot con consulta de tickets
# AI Elite, S.R.L.
# ============================================================

# difflib permite comparar textos y buscar coincidencias aproximadas.
# Se usa cuando el usuario escribe una consulta que no coincide exactamente
# con una palabra de la base de conocimiento.
import difflib

# re permite trabajar con expresiones regulares.
# En este caso se usa para detectar números de ticket dentro de la consulta.
import re


# ============================================================
# Base de conocimiento del chatbot
# ============================================================

# Este diccionario contiene palabras clave y respuestas asociadas.
# Cuando el usuario escribe una consulta, el chatbot busca si alguna
# de estas palabras aparece en el mensaje.
base_conocimiento = {
    "hola": "Hola. Bienvenido al sistema inteligente de soporte de AI Elite, S.R.L. ¿En qué puedo ayudarte?",
    "prioridad": "La prioridad indica el nivel de atención que debe recibir una solicitud. Puede ser Baja, Media o Alta.",
    "ticket": "Un ticket es el registro que almacena toda la información relacionada con una solicitud de soporte.",
    "categoria": "Las categorías permiten clasificar las incidencias, por ejemplo: correo electrónico, internet, impresoras, acceso al sistema o bases de datos.",
    "escalar": "Escalar un ticket significa enviarlo a un nivel técnico superior cuando requiere atención especializada.",
    "agente": "El agente inteligente analiza la urgencia, el impacto, el horario y la necesidad de escalamiento para recomendar una acción.",
    "machine learning": "El modelo de Machine Learning aprende a predecir la prioridad de una solicitud utilizando ejemplos del dataset.",
    "soporte": "El departamento de soporte técnico atiende las incidencias reportadas por los usuarios de AI Elite, S.R.L.",
    "gracias": "Con gusto. Estoy disponible para ayudarte cuando lo necesites."
}


# ============================================================
# Función para consultar tickets desde el chatbot
# ============================================================

def consultar_ticket_chatbot(pregunta):
    """
    Esta función revisa si la pregunta del usuario contiene un número.
    Si encuentra un número, lo interpreta como posible número de ticket
    y busca la información correspondiente en el dataset.
    """

    # Buscar números dentro del texto escrito por el usuario.
    numeros = re.findall(r"\d+", pregunta)

    # Si no hay números, significa que la consulta no es sobre un ticket específico.
    if not numeros:
        return None

    # Tomar el primer número encontrado y convertirlo a entero.
    numero_ticket = int(numeros[0])

    # Buscar el ticket dentro del dataset.
    resultado = df[df["ticket"] == numero_ticket]

    # Si el ticket no existe, se informa al usuario.
    if resultado.empty:
        return f"No encontré ningún ticket registrado con el número {numero_ticket}."

    # Si el ticket existe, se toma la primera coincidencia encontrada.
    ticket = resultado.iloc[0]

    # Convertir la pregunta a minúsculas para facilitar la comparación.
    pregunta_limpia = pregunta.lower()

    # Responder solo con la prioridad si el usuario pregunta por prioridad.
    if "prioridad" in pregunta_limpia:
        return f"La prioridad del ticket {numero_ticket} es {ticket['prioridad']}."

    # Responder sobre escalamiento si el usuario pregunta si debe escalarse.
    if "escalar" in pregunta_limpia or "escala" in pregunta_limpia:
        return f"El ticket {numero_ticket} tiene escalamiento marcado como: {ticket['escalar']}."

    # Responder con la acción recomendada si el usuario pregunta por acción.
    if "accion" in pregunta_limpia or "acción" in pregunta_limpia or "recomienda" in pregunta_limpia:
        return f"La acción recomendada para el ticket {numero_ticket} es: {ticket['accion_recomendada']}."

    # Responder con la categoría si el usuario pregunta por categoría.
    if "categoria" in pregunta_limpia or "categoría" in pregunta_limpia:
        return f"La categoría del ticket {numero_ticket} es: {ticket['categoria']}."

    # Si el usuario pregunta por estado o información general, se muestra un resumen.
    if "estado" in pregunta_limpia or "informacion" in pregunta_limpia or "información" in pregunta_limpia or "ticket" in pregunta_limpia:
        return (
            f"Información del ticket {numero_ticket}:\n"
            f"- Categoría: {ticket['categoria']}\n"
            f"- Área: {ticket['area']}\n"
            f"- Urgencia: {ticket['urgencia']}\n"
            f"- Impacto: {ticket['impacto']}\n"
            f"- Prioridad: {ticket['prioridad']}\n"
            f"- Escalar: {ticket['escalar']}\n"
            f"- Acción recomendada: {ticket['accion_recomendada']}"
        )

    # Respuesta general cuando se encuentra el ticket, pero no se identifica
    # una pregunta específica sobre prioridad, categoría, acción o escalamiento.
    return (
        f"Encontré el ticket {numero_ticket}. "
        f"Su prioridad es {ticket['prioridad']} y la acción recomendada es: "
        f"{ticket['accion_recomendada']}."
    )


# ============================================================
# Función principal para obtener una respuesta
# ============================================================

def obtener_respuesta(pregunta):
    """
    Esta función decide cómo responder al usuario.
    Primero verifica si la consulta contiene un número de ticket.
    Si no es una consulta de ticket, busca una respuesta en la base de conocimiento.
    """

    # Normalizar la pregunta del usuario.
    pregunta_limpia = pregunta.lower()

    # Primero se intenta responder como consulta de ticket.
    respuesta_ticket = consultar_ticket_chatbot(pregunta_limpia)

    if respuesta_ticket:
        return respuesta_ticket

    # Buscar coincidencias directas en la base de conocimiento.
    for clave, respuesta in base_conocimiento.items():
        if clave in pregunta_limpia:
            return respuesta

    # Si no hay coincidencia directa, se intenta una coincidencia aproximada.
    coincidencias = difflib.get_close_matches(
        pregunta_limpia,
        base_conocimiento.keys(),
        n=1,
        cutoff=0.5
    )

    # Si existe una coincidencia aproximada, se devuelve la respuesta asociada.
    if coincidencias:
        return base_conocimiento[coincidencias[0]]

    # Respuesta alternativa cuando el chatbot no entiende la consulta.
    return (
        "Lo siento, todavía no tengo información suficiente para responder esa consulta. "
        "Puedes preguntarme sobre tickets, prioridades, escalamiento, categorías, soporte técnico, agente inteligente o Machine Learning."
    )


# ============================================================
# Función para iniciar la conversación
# ============================================================

def iniciar_chat():
    """
    Esta función inicia el chatbot, muestra las instrucciones
    y mantiene la conversación activa hasta que el usuario escriba
    una palabra de salida.
    """

    # Mensaje de bienvenida.
    print("=" * 70)
    print("                 AI Elite, S.R.L.")
    print("             Asistente Virtual SupportBot")
    print("=" * 70)

    print("""
Soy un asistente virtual diseñado para responder consultas
relacionadas con el sistema de soporte técnico.

Puedes realizar preguntas como:

• ¿Qué es un ticket?
• ¿Qué significa prioridad?
• ¿Cuál es el estado del ticket 1007?
• ¿Qué prioridad tiene el ticket 1007?
• ¿Debe escalarse el ticket 1007?
• ¿Qué acción recomienda el ticket 1007?
• ¿Qué hace el agente inteligente?

Para finalizar escribe:

salir
adiós
finalizar
""")

    # Ciclo principal de conversación.
    while True:
        pregunta = input("Escribe tu consulta: ")

        # Condición para finalizar la conversación.
        if pregunta.lower() in ["salir", "adios", "adiós", "finalizar"]:
            print("\nGracias por utilizar SupportBot.")
            print("Hasta luego.")
            break

        # Mostrar la respuesta generada por el chatbot.
        print("\nSupportBot:")
        print(obtener_respuesta(pregunta))
        print()

In [ ]:
iniciar_chat()

# Modelo de Machine Learning para predecir la prioridad

En esta sección se entrena un modelo básico de Machine Learning para predecir la prioridad de una solicitud de soporte técnico.

La variable objetivo será **prioridad**, ya que el sistema intentará clasificar cada solicitud como:

- Baja
- Media
- Alta

Para realizar esta predicción se utilizarán variables del dataset como categoría, área, urgencia, impacto, tiempo estimado, canal, tipo de usuario, dispositivo y horario.

Como la mayoría de estas variables son categóricas, será necesario convertirlas a formato numérico antes de entrenar el modelo.

In [ ]:
# Selección de variables de entrada
variables_entrada = [
    "categoria",
    "area",
    "urgencia",
    "impacto",
    "tiempo_estimado_min",
    "canal",
    "tipo_usuario",
    "dispositivo",
    "horario"
]

# Variable objetivo
variable_objetivo = "prioridad"

X = df[variables_entrada]
y = df[variable_objetivo]

print("Variables de entrada seleccionadas:")
print(variables_entrada)

print("\nVariable objetivo:")
print(variable_objetivo)

## Preparación de los datos

El modelo de Machine Learning no puede trabajar directamente con textos como "Alta", "Media", "Correo" o "Internet". Por esta razón, se utiliza una técnica llamada codificación de variables categóricas, que convierte los valores de texto en valores numéricos.

En esta práctica se emplea `LabelEncoder` para transformar las columnas categóricas del dataset.

In [ ]:
# Crear copia de las variables de entrada para no alterar el dataset original
X_codificado = X.copy()

# Diccionario para guardar los codificadores utilizados
codificadores = {}

# Convertir columnas categóricas a valores numéricos
for columna in X_codificado.columns:
    if X_codificado[columna].dtype == "object":
        le = LabelEncoder()
        X_codificado[columna] = le.fit_transform(X_codificado[columna])
        codificadores[columna] = le

# Codificar la variable objetivo
codificador_objetivo = LabelEncoder()
y_codificado = codificador_objetivo.fit_transform(y)

X_codificado.head()

## División entre entrenamiento y prueba

Después de preparar los datos, se divide el dataset en dos partes:

- Datos de entrenamiento: se usan para que el modelo aprenda.
- Datos de prueba: se usan para evaluar qué tan bien funciona el modelo con información que no vio durante el entrenamiento.

Esta separación evita evaluar el modelo con los mismos datos que utilizó para aprender.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_codificado,
    y_codificado,
    test_size=0.25,
    random_state=0
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

## Entrenamiento del modelo

Para esta práctica se utiliza un modelo de Árbol de Decisión, recomendado en la guía del proyecto. Este algoritmo permite clasificar los casos aplicando divisiones sucesivas sobre las variables de entrada hasta llegar a una predicción.

In [ ]:
modelo_prioridad = DecisionTreeClassifier(random_state=0)

modelo_prioridad.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

## Evaluación del modelo

Para evaluar el desempeño del modelo se utiliza la métrica `accuracy`, que indica el porcentaje de predicciones correctas realizadas sobre los datos de prueba.

In [ ]:
predicciones = modelo_prioridad.predict(X_test)

accuracy = accuracy_score(y_test, predicciones)

print("Accuracy del modelo:", round(accuracy, 4))
print("Porcentaje de acierto:", round(accuracy * 100, 2), "%")

In [ ]:
print("Reporte de clasificación:")
print(
    classification_report(
        y_test,
        predicciones,
        target_names=codificador_objetivo.classes_
    )
)

In [ ]:
print("Matriz de confusión:")
print(confusion_matrix(y_test, predicciones))

### Interpretación del modelo

El modelo de Árbol de Decisión permitió predecir la prioridad de las solicitudes utilizando variables como categoría, área, urgencia, impacto, canal, tipo de usuario, dispositivo y horario.

El valor de `accuracy` indica el porcentaje de casos del conjunto de prueba que fueron clasificados correctamente. Mientras mayor sea este valor, mejor será el desempeño general del modelo. Aun así, debido a que el dataset es simulado y contiene una cantidad limitada de registros, el resultado debe interpretarse como una demostración académica y no como una solución lista para implementarse en una empresa real.